In [0]:
%sql
USE uc_quickbite.gold_aggregate;

1.Monthly Orders Trend

In [0]:
%sql
--Compare Pre-Crisis vs Crisis vs Recovery

CREATE OR REPLACE TABLE monthly_orders_trend AS
SELECT
  date_format(order_timestamp, 'yyyy-MM') AS order_month,
  crisis_phase,
  COUNT(order_id) AS total_orders,
  SUM(net_revenue) AS total_revenue
FROM uc_quickbite.silver_transform.orders_enriched
GROUP BY 1,2
ORDER BY 1,2;

2. City-wise Order Decline

In [0]:
%sql
CREATE OR REPLACE TABLE city_order_decline AS
WITH base AS (
  SELECT
    customer_city,
    crisis_phase,
    COUNT(order_id) AS orders_cnt
  FROM uc_quickbite.silver_transform.orders_enriched
  GROUP BY customer_city, crisis_phase
)
SELECT
  customer_city,
  SUM(CASE WHEN crisis_phase = 'Pre-Crisis' THEN orders_cnt ELSE 0 END) AS pre_crisis_orders,
  SUM(CASE WHEN crisis_phase = 'Crisis' THEN orders_cnt ELSE 0 END) AS crisis_orders,
  ROUND(
    (SUM(CASE WHEN crisis_phase = 'Crisis' THEN orders_cnt ELSE 0 END)
    - SUM(CASE WHEN crisis_phase = 'Pre-Crisis' THEN orders_cnt ELSE 0 END))
    * 100.0
    / NULLIF(SUM(CASE WHEN crisis_phase = 'Pre-Crisis' THEN orders_cnt ELSE 0 END), 0),
  2) AS pct_order_decline
FROM base
GROUP BY customer_city
ORDER BY pct_order_decline;


3. Restaurant Volume Decline

In [0]:
%sql
CREATE OR REPLACE TABLE restaurant_order_decline AS
WITH rest_orders AS (
  SELECT
    restaurant_id,
    restaurant_name,
    crisis_phase,
    COUNT(order_id) AS order_cnt
  FROM uc_quickbite.silver_transform.orders_enriched
  GROUP BY restaurant_id, restaurant_name, crisis_phase
),
pivoted AS (
  SELECT
    restaurant_id,
    restaurant_name,
    SUM(CASE WHEN crisis_phase='Pre-Crisis' THEN order_cnt ELSE 0 END) AS pre_orders,
    SUM(CASE WHEN crisis_phase='Crisis' THEN order_cnt ELSE 0 END) AS crisis_orders
  FROM rest_orders
  GROUP BY restaurant_id, restaurant_name
)
SELECT
  *,
  ROUND((crisis_orders - pre_orders) * 100.0 / NULLIF(pre_orders,0),2) AS pct_decline
FROM pivoted
WHERE pre_orders >= 50
ORDER BY pct_decline;


4. Cancellation Trends by City

In [0]:
%sql
CREATE OR REPLACE TABLE cancellation_trends AS
SELECT
  customer_city,
  crisis_phase,
  COUNT(order_id) AS total_orders,
  SUM(CASE WHEN is_cancelled='Y' THEN 1 ELSE 0 END) AS cancelled_orders,
  ROUND(
    SUM(CASE WHEN is_cancelled='Y' THEN 1 ELSE 0 END) * 100.0
    / COUNT(order_id),
  2) AS cancellation_rate
FROM uc_quickbite.silver_transform.orders_enriched
GROUP BY customer_city, crisis_phase;


5. Delivery SLA Metrics

In [0]:
%sql
CREATE OR REPLACE TABLE delivery_sla_summary AS
SELECT
  crisis_phase,
  ROUND(AVG(actual_delivery_time_mins),2) AS avg_delivery_time,
  ROUND(AVG(delivery_delay_mins),2) AS avg_delay_mins,
  ROUND(AVG(sla_breached) * 100,2) AS sla_breach_pct
FROM uc_quickbite.silver_transform.orders_enriched
GROUP BY crisis_phase;

6. Monthly Ratings Trend

In [0]:
%sql
CREATE OR REPLACE TABLE monthly_ratings_trend AS
SELECT
  date_format(
    to_timestamp(review_timestamp, 'dd-MM-yyyy HH:mm'),
    'yyyy-MM'
  ) AS review_month,
  ROUND(AVG(rating),2) AS avg_rating,
  COUNT(order_id) AS total_reviews
FROM uc_quickbite.silver_transform.ratings_enriched
GROUP BY 1
ORDER BY 1;

7. Sentiment Summary

In [0]:
%sql
CREATE OR REPLACE TABLE sentiment_summary AS
SELECT
  sentiment_bucket,
  COUNT(*) AS review_count
FROM uc_quickbite.silver_transform.ratings_enriched
GROUP BY sentiment_bucket;

8. Revenue Impact Analysis

In [0]:
%sql
CREATE OR REPLACE TABLE revenue_impact AS
SELECT
  crisis_phase,
  ROUND(SUM(net_revenue),2) AS total_net_revenue,
  COUNT(order_id) AS total_orders
FROM uc_quickbite.silver_transform.orders_enriched
GROUP BY crisis_phase;

9. Loyalty Churn

In [0]:
%sql
CREATE OR REPLACE TABLE loyalty_churn AS
WITH loyal_customers AS (
  SELECT
    customer_id
  FROM uc_quickbite.silver_transform.orders_enriched
  WHERE crisis_phase='Pre-Crisis'
  GROUP BY customer_id
  HAVING COUNT(order_id) >= 5
),
crisis_orders AS (
  SELECT DISTINCT customer_id
  FROM uc_quickbite.silver_transform.orders_enriched
  WHERE crisis_phase='Crisis'
)
SELECT
  l.customer_id,
  CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END AS churned_during_crisis
FROM loyal_customers l
LEFT JOIN crisis_orders c
  ON l.customer_id = c.customer_id;


10. High-Value Customer Decline

In [0]:
%sql
CREATE OR REPLACE TABLE high_value_customer_decline AS
WITH customer_spend AS (
  SELECT
    customer_id,
    SUM(net_revenue) AS total_spend
  FROM uc_quickbite.silver_transform.orders_enriched
  WHERE crisis_phase='Pre-Crisis'
  GROUP BY customer_id
),
top_customers AS (
  SELECT
    *,
    NTILE(20) OVER (ORDER BY total_spend DESC) AS spend_bucket
  FROM customer_spend
)
SELECT
  t.customer_id,
  t.total_spend,
  COUNT(o.order_id) AS crisis_orders,
  ROUND(AVG(o.delivery_delay_mins),2) AS avg_delay,
  ROUND(AVG(r.rating),2) AS avg_rating
FROM top_customers t
LEFT JOIN uc_quickbite.silver_transform.orders_enriched o
  ON t.customer_id = o.customer_id
LEFT JOIN uc_quickbite.silver_transform.ratings_enriched r
  ON o.order_id = r.order_id
WHERE t.spend_bucket = 1
GROUP BY t.customer_id, t.total_spend;


GOLD LAYER PERFORMANCE OPTIMIZATION

In [0]:
%sql
OPTIMIZE uc_quickbite.gold_aggregate.monthly_orders_trend ZORDER BY (order_month);

In [0]:
%sql
OPTIMIZE uc_quickbite.gold_aggregate.city_order_decline ZORDER BY (customer_city);

In [0]:
%sql
OPTIMIZE uc_quickbite.gold_aggregate.restaurant_order_decline ZORDER BY (restaurant_id);